# File Dokumen: Data Cleaning Cuaca Bandung.ipynb
file ini berisi tentang data cleaning cuaca di Bandung. Data ini mencakup informasi tentang suhu, kelembapan, curah hujan, dan lainnya. Tujuan dari data cleaning ini adalah untuk membersihkan dan mempersiapkan data for analisis lebih lanjut. Proses data cleaning melibatkan langkah-langkah seperti menghapus duplikasi, menangani nilai yang hilang, memperbaiki format data, dan mengidentifikasi serta menangani outlier. Data yang telah dibersihkan akan lebih akurat dan dapat diandalkan for analisis tren cuaca, perbandingan kondisi cuaca antar tanggal, dan prediksi cuaca di masa depan. Data ini dapat digunakan oleh peneliti, pengamat cuaca, dan pembuat kebijakan for memahami dinamika cuaca di Bandung dan membuat keputusan yang lebih baik.

In [1]:
import os
import pandas as pd

path1 = 'dataset/Gabungan_Laporan_Iklim_Harian_Bandung_Mei_2024_April_2026.csv'
path2 = '../dataset/Gabungan_Laporan_Iklim_Harian_Bandung_Mei_2024_April_2026.csv'

if os.path.exists(path1):
    path = path1
elif os.path.exists(path2):
    path = path2
else:
    raise FileNotFoundError("Neither file path exists.")

df_cuacabandung = pd.read_csv(path)
df_cuacabandung.info()

<class 'pandas.DataFrame'>
RangeIndex: 727 entries, 0 to 726
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   TANGGAL  727 non-null    str    
 1   TN       726 non-null    float64
 2   TX       727 non-null    float64
 3   TAVG     726 non-null    float64
 4   RH_AVG   726 non-null    float64
 5   RR       711 non-null    float64
 6   SS       726 non-null    float64
 7   FF_X     727 non-null    float64
 8   DDD_X    727 non-null    float64
dtypes: float64(8), str(1)
memory usage: 51.2 KB


In [2]:
df_cuacabandung.head(10)

,TANGGAL,TN,TX,TAVG,RH_AVG,RR,SS,FF_X,DDD_X
0,01-01-2025,21.6,32.8,25.0,75.0,8888.0,1.3,4.0,270.0
1,02-01-2025,21.2,33.4,25.9,60.0,3.5,6.5,6.0,233.0
2,03-01-2025,21.2,32.8,26.3,73.0,0.3,8.0,3.0,290.0
3,04-01-2025,22.0,31.8,25.0,75.0,0.0,8.0,3.0,260.0
4,05-01-2025,22.0,30.6,25.6,72.0,3.2,5.3,2.0,260.0
5,06-01-2025,20.6,32.4,24.5,78.0,0.0,3.5,3.0,248.0
6,07-01-2025,20.2,31.0,28.2,69.0,26.9,5.7,5.0,270.0
7,08-01-2025,20.6,30.4,24.9,79.0,0.0,8.0,3.0,300.0
8,09-01-2025,22.2,26.4,23.2,86.0,2.2,3.5,3.0,300.0
9,10-01-2025,21.8,31.0,25.3,79.0,8888.0,0.0,5.0,270.0


# Data Cleaning

In [3]:
import numpy as np

# 1. Mengubah format kolom TANGGAL menjadi Datetime
df_cuacabandung['TANGGAL'] = pd.to_datetime(df_cuacabandung['TANGGAL'], format='%d-%m-%Y')

# 2. Mengganti nilai 8888 dan 9999 dengan NaN (Not a Number)
df_cuacabandung = df_cuacabandung.replace([8888, 9999, 8888.0, 9999.0], np.nan)

# Cek kembali jumlah missing values setelah direplace
print("Jumlah Missing Values sebelum di-handle:")
print(df_cuacabandung.isnull().sum())

Jumlah Missing Values sebelum di-handle:
TANGGAL     0
TN          1
TX          0
TAVG        1
RH_AVG      1
RR         87
SS          1
FF_X        0
DDD_X       0
dtype: int64


**Penjelasan:**
Pada tahap *Data Cleaning* pertama ini:
1. Kolom `TANGGAL` diubah formatnya menjadi tipe `datetime` agar bisa digunakan untuk analisis deret waktu (*time series*).
2. Nilai `8888.0` (data tidak terukur) dan `9999.0` (tidak ada data) diubah menjadi `NaN` (*Not a Number*), karena nilai tersebut adalah representasi dari data yang hilang (*missing value*).
3. Terakhir, mengecek jumlah *missing values* pada setiap kolom setelah nilai yang tidak valid diganti.\

In [4]:
# 3. Cek duplikat tanggal
print("Jumlah duplikat tanggal:", df_cuacabandung['TANGGAL'].duplicated().sum())


Jumlah duplikat tanggal: 0


In [5]:
# 4. Interpolasi dengan batasan
df_cuacabandung = df_cuacabandung.interpolate(method='linear', limit=5, limit_direction='both')

# 5. Set index
df_cuacabandung.set_index('TANGGAL', inplace=True)
df_cuacabandung.sort_index(inplace=True)

print("\nJumlah Missing Values setelah interpolasi:")
print(df_cuacabandung.isnull().sum())
df_cuacabandung.head()


Jumlah Missing Values setelah interpolasi:
TN        0
TX        0
TAVG      0
RH_AVG    0
RR        0
SS        0
FF_X      0
DDD_X     0
dtype: int64


,TN,TX,TAVG,RH_AVG,RR,SS,FF_X,DDD_X
TANGGAL,,,,,,,,
2024-05-01,21.8,30.0,26.1,81.0,2.5,5.2,2.0,240.0
2024-05-02,21.8,32.2,26.1,73.0,5.0,6.0,2.0,120.0
2024-05-03,22.0,32.0,26.2,73.0,3.9,5.3,2.0,80.0
2024-05-04,21.4,31.6,25.1,76.0,2.8,6.7,3.0,120.0
2024-05-05,21.2,31.2,25.5,72.0,1.7,6.4,5.0,220.0


In [6]:
print("Index name:", df_cuacabandung.index.name)
print("Columns:", df_cuacabandung.columns.tolist())
print("\nData setelah cleaning:")
df_cuacabandung.head()

Index name: TANGGAL
Columns: ['TN', 'TX', 'TAVG', 'RH_AVG', 'RR', 'SS', 'FF_X', 'DDD_X']

Data setelah cleaning:


,TN,TX,TAVG,RH_AVG,RR,SS,FF_X,DDD_X
TANGGAL,,,,,,,,
2024-05-01,21.8,30.0,26.1,81.0,2.5,5.2,2.0,240.0
2024-05-02,21.8,32.2,26.1,73.0,5.0,6.0,2.0,120.0
2024-05-03,22.0,32.0,26.2,73.0,3.9,5.3,2.0,80.0
2024-05-04,21.4,31.6,25.1,76.0,2.8,6.7,3.0,120.0
2024-05-05,21.2,31.2,25.5,72.0,1.7,6.4,5.0,220.0


**Penjelasan:**
Tahap *Data Cleaning* dilanjutkan dengan:
1. Mengatasi nilai kosong (*missing values*) menggunakan metode **interpolasi linier**. Metode ini sangat cocok untuk data cuaca karena dapat memperkirakan nilai yang hilang berdasarkan nilai sebelum dan sesudahnya dengan asumsi perubahan yang konstan.
2. Kolom `TANGGAL` diatur sebagai *index* (*set_index*) pada DataFrame. Penggunaan waktu sebagai *index* merupakan standar dalam pengolahan data *time series*, sehingga mempermudah proses plotting maupun pemotongan data (*slicing*) berdasarkan rentang waktu tertentu.\

In [7]:
# save data yang sudah dibersihkan ke file baru
df_cuacabandung.to_csv('../dataset/dataset_cuaca_bandung_cleaned.csv', index=True)
print("Data cuaca Bandung berhasil disimpan di ../dataset/dataset_cuaca_bandung_cleaned.csv")

Data cuaca Bandung berhasil disimpan di ../dataset/dataset_cuaca_bandung_cleaned.csv


In [8]:
df_cuacabandung.head()

,TN,TX,TAVG,RH_AVG,RR,SS,FF_X,DDD_X
TANGGAL,,,,,,,,
2024-05-01,21.8,30.0,26.1,81.0,2.5,5.2,2.0,240.0
2024-05-02,21.8,32.2,26.1,73.0,5.0,6.0,2.0,120.0
2024-05-03,22.0,32.0,26.2,73.0,3.9,5.3,2.0,80.0
2024-05-04,21.4,31.6,25.1,76.0,2.8,6.7,3.0,120.0
2024-05-05,21.2,31.2,25.5,72.0,1.7,6.4,5.0,220.0


In [9]:
df_cuacabandung.tail()

,TN,TX,TAVG,RH_AVG,RR,SS,FF_X,DDD_X
TANGGAL,,,,,,,,
2026-04-23,21.8,29.2,24.9,85.0,0.2,4.5,2.0,280.0
2026-04-24,20.6,31.0,25.0,78.0,0.0,4.5,3.0,270.0
2026-04-25,20.8,32.0,25.1,74.0,0.0,7.2,3.0,40.0
2026-04-26,20.4,30.4,25.0,77.0,0.0,6.7,4.0,287.0
2026-04-27,21.6,31.4,25.4,81.0,3.4,7.2,4.0,290.0
